# 02 · SSL pretraining (SimCLR, P_ecg mid, seed 0)

**Runtime → Change runtime type → T4 GPU.** Then **Runtime → Run all**.

Pretrains the xresnet1d50 encoder on PTB-XL folds 1–8 **without labels** (`docs/ssl_protocol_v1.md`).
The run checkpoints to Drive every 5 epochs. If Colab disconnects, open the notebook again and **Run all**:
the timing cell is skipped and pretraining resumes from the last checkpoint.

Do not use *File → Save a copy in GitHub*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, zipfile, json
PROJECT = '/content/drive/MyDrive/robust-ecg-ssl'
CACHE_DRIVE, RUNS, SSL = f'{PROJECT}/cache', f'{PROJECT}/runs', f'{PROJECT}/ssl'
CACHE = '/content/cache'                      # local copy: much faster reads than Drive
REPO = '/content/robust-ecg-ssl'
os.makedirs(SSL, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Salako07/robust-ecg-ssl.git {REPO}
!git -C {REPO} pull -q && git -C {REPO} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# copy the cache to local disk once per session (~1-3 min)
os.makedirs(CACHE, exist_ok=True)
for f in os.listdir(CACHE_DRIVE):
    s, d = f'{CACHE_DRIVE}/{f}', f'{CACHE}/{f}'
    if os.path.isfile(s) and (not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s)):
        shutil.copy(s, d)
!ls -la {CACHE}

## Timing check
30 steps at batch 512 and a projection for 300 epochs. Skipped once the run has started.
If it reports **out of memory**, stop and paste the error in the chat (do not change the batch size here:
batch size is part of the frozen protocol and any change has to be logged).

In [ ]:
RUN = 'SSL_ecg-mid_s0'
if not os.path.exists(f'{SSL}/{RUN}/ckpt_last.pt') and not os.path.exists(f'{SSL}/{RUN}/done.json'):
    !python {REPO}/scripts/pretrain_simclr.py --policy ecg --strength mid --seed 0 --cache {CACHE} --runs {SSL} --max-steps 30
    !nvidia-smi --query-gpu=memory.used --format=csv
else:
    print('run already started, timing skipped')

## Pretraining
Prints one line per epoch: NT-Xent loss, in-batch retrieval top-1 (chance ≈ 0.001), learning rate, minutes.

In [ ]:
!python {REPO}/scripts/pretrain_simclr.py --policy ecg --strength mid --seed 0 --cache {CACHE} --runs {SSL} --workers 2

In [ ]:
import pandas as pd
print(json.dumps(json.load(open(f'{SSL}/{RUN}/done.json')), indent=1))
log = pd.read_csv(f'{SSL}/{RUN}/log.csv')
print(log.iloc[[0, 1, 4, 9, 19, 49, 99, 149, 199, 249, -1]].round(4).to_string(index=False))

## First matched pair at 100% labels (seed 0)
Two runs of the core grid, ~6 min each: S1 (random init + P_ecg) and C1 (this encoder + P_ecg).
Together with S0 they give the first single-seed look at C1 − S1. **One seed, no confidence interval:
not evidence for or against any hypothesis.**

In [ ]:
!python {REPO}/scripts/train.py --arm S1 --budget 1.0 --seed 0 --cache {CACHE} --runs {RUNS} --workers 2
!python {REPO}/scripts/train.py --arm C1 --budget 1.0 --seed 0 --cache {CACHE} --runs {RUNS} --workers 2 --pretrained {SSL}/{RUN}/encoder.pt
keys = ['test_macro_auroc', 'best_val_macro_auroc', 'best_step', 'test_e3_macro_auroc', 'sph_e3_macro_auroc', 'e3_degradation']
rows = {}
for rid in ['S0_b1_s0_lik0', 'S1_b1_s0_lik0', 'C1_b1_s0_lik0']:
    p = f'{RUNS}/{rid}/done.json'
    if os.path.exists(p):
        r = json.load(open(p)); rows[rid] = {k: r[k] for k in keys} | {k[10:]: r[k] for k in r if k.startswith('sph_auroc_')}
print(pd.DataFrame(rows).round(4).to_string())

## Export results for the repository
Creates `results_export.zip` on Drive with every finished run's small files (fine-tuning runs and SSL runs;
`encoder.pt` and checkpoints are left out, their hashes are in the JSON files). **Download it and attach it in the chat.**

In [ ]:
out = f'{PROJECT}/results_export.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, arc in ((RUNS, 'runs'), (SSL, 'ssl')):
        for f in glob.glob(f'{root}/*/*'):
            if f.endswith(('config.json', 'log.csv', 'done.json', 'predictions.npz')) and \
               os.path.exists(os.path.join(os.path.dirname(f), 'done.json')):
                z.write(f, os.path.join(arc, os.path.relpath(f, root)))
    for reg, arc in ((f'{RUNS}/registry.csv', 'runs/registry.csv'), (f'{SSL}/registry_ssl.csv', 'ssl/registry_ssl.csv')):
        if os.path.exists(reg):
            z.write(reg, arc)
print(out, round(os.path.getsize(out) / 1e6, 2), 'MB')
print(zipfile.ZipFile(out).namelist())

**Paste the outputs of the timing cell, the summary cell and the matched-pair cell into the chat, and attach the zip.**